In [1]:
from glob import glob
import pandas as pd
import os
import soundfile as sf
from tqdm import tqdm
from multiprocess import Pool
from scipy.io import wavfile
import itertools
import io
import numpy as np
import json
import re
import zipfile
from pathlib import Path

def chunks(l, n):
    for i in range(0, len(l), n):
        yield (l[i: i + n], i // n)

def multiprocessing(strings, function, cores=6, returned=True):
    df_split = chunks(strings, len(strings) // cores)
    pool = Pool(cores)
    pooled = pool.map(function, df_split)
    pool.close()
    pool.join()

    if returned:
        return list(itertools.chain(*pooled))

/usr/lib/python3/dist-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [3]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="SparkAudio/voxbox", 
    repo_type="dataset", 
    local_dir="./hq-conversations",
    allow_patterns="audios/hq-conversations/*.tar.gz"
)

Fetching 1 files: 100%|██████████| 1/1 [00:06<00:00,  6.77s/it]


'/home/ubuntu/hq-conversations'

In [10]:
# import tarfile

# def loop(files):
#     files, _ = files
#     for f in tqdm(files):
#         try:
#             with tarfile.open(f, "r:gz") as tar:
#                 tar.extractall(path='hq-conversations')
#             os.remove(f)
#         except Exception as e:
#             print(e)

In [11]:
# files = glob('hq-conversations/audios/hq-conversations/*.tar.gz')
# multiprocessing(files, loop, len(files))

In [23]:
# !wget https://huggingface.co/datasets/SparkAudio/voxbox/resolve/main/metadata/hq-conversations.jsonl
# !wget https://huggingface.co/datasets/SparkAudio/voxbox/raw/main/speaker_ids/hq-conversations.txt

In [28]:
with open('hq-conversations.txt') as fopen:
    speakers = fopen.read().split('\n')
speakers = [s for s in speakers if '\t' in s]
speakers_mapping = {}
for s in speakers:
    l, r = s.split('\t')
    speakers_mapping[l] = r
len(speakers_mapping)

50983

In [32]:
rows = []
with open('hq-conversations.jsonl') as fopen:
    for l in fopen:
        l = json.loads(l)
        rows.append(l)
len(rows)

50982

In [31]:
def loop(rows):
    rows, _ = rows
    data = []
    base = 'hq-conversations_audio'
    os.makedirs(base, exist_ok=True)
    for row in tqdm(rows):
        
        if row['index'] not in speakers_mapping:
            continue
            
        speaker = speakers_mapping[row['index']]

        if not os.path.exists(row['wav_path']):
            continue

        t = row['text'].strip()
        if len(t) < 2:
            continue

        audio_filename = os.path.join(base, row['wav_path'].replace('.wav', '.mp3').replace('/', '-'))
        audio_np, sr = sf.read(row['wav_path'])
        if audio_np.ndim > 1:
            audio_np = audio_np.mean(axis=1)
        if audio_np.shape[0] < 10000:
            continue
        sf.write(audio_filename, audio_np, sr)
        
        data.append({
            'audio_filename': audio_filename,
            'text': t,
            'speaker': f"{base}_{speaker}"
        })
    return data

In [33]:
data = loop((rows[:10], 0))

100%|██████████| 10/10 [00:00<00:00, 17.08it/s]


In [35]:
data = multiprocessing(rows, loop, cores = 20)

100%|██████████| 2549/2549 [02:19<00:00, 18.25it/s]


In [36]:
from datasets import Dataset

dataset = Dataset.from_list(data)
dataset[0]

{'audio_filename': 'hq-conversations_audio/hq-conversations-HQ-Conversations_0000000000.mp3',
 'text': '唉，你今年过年被催婚了吗？',
 'speaker': 'hq-conversations_audio_G0007'}

In [37]:
dataset.push_to_hub('malaysia-ai/Multilingual-TTS', 'hq-conversations')

Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 43.70ba/s]
Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (1 / 1): 100%|██████████| 3.61MB / 3.61MB,  392kB/s  
Processing Files (1 / 1): 100%|██████████| 3.61MB / 3.61MB,  384kB/s  
New Data Upload: 100%|██████████| 3.61MB / 3.61MB,  384kB/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:09<00:00,  9.77s/ shards]


CommitInfo(commit_url='https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS/commit/1ec5041fc23250a85ad1385e4f14db1536fa4d5b', commit_message='Upload dataset', commit_description='', oid='1ec5041fc23250a85ad1385e4f14db1536fa4d5b', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS', endpoint='https://huggingface.co', repo_type='dataset', repo_id='malaysia-ai/Multilingual-TTS'), pr_revision=None, pr_num=None)

In [38]:
audio_files = [d['audio_filename'] for d in data]

with open('hq-conversations-audio.json', 'w') as fopen:
    json.dump(list(set(audio_files)), fopen)

In [41]:
# !zip -rq hq-conversations_audio.zip hq-conversations_audio

In [42]:
# !hf upload malaysia-ai/Multilingual-TTS hq-conversations_audio.zip --repo-type=dataset

In [45]:
# !zip -rq hq-conversations_audio_neucodec.zip hq-conversations_audio_neucodec

In [44]:
# !hf upload malaysia-ai/Multilingual-TTS hq-conversations_audio_neucodec.zip --repo-type=dataset

Processing Files (0 / 0)      : |                  |  0.00B /  0.00B            
New Data Upload               : |                  |  0.00B /  0.00B            

  ...ations_audio_neucodec.zip:  99%|█████████████▉| 67.9MB / 68.5MB            

Processing Files (0 / 1)      :  99%|█████████████▉| 67.9MB / 68.5MB, 6.66MB/s  
New Data Upload               :  99%|█████████████▉| 67.9MB / 68.5MB, 6.66MB/s  

  ...ations_audio_neucodec.zip:  99%|█████████████▉| 67.9MB / 68.5MB            

  ...ations_audio_neucodec.zip:  99%|█████████████▉| 67.9MB / 68.5MB            

  ...ations_audio_neucodec.zip:  99%|█████████████▉| 67.9MB / 68.5MB            

Processing Files (1 / 1)      : 100%|██████████████| 68.5MB / 68.5MB, 6.72MB/s  
New Data Upload               : 100%|██████████████| 68.5MB / 68.5MB, 6.72MB/s  

  ...ations_audio_neucodec.zip: 100%|██████████████| 68.5MB / 68.5MB            

  ...ations_audio_neucodec.zip: 100%|██████████████| 68.5MB / 68.5MB            

Processing Files (1